# V12 target sharpening pilot (Path B v2 → sharpening pivot)

## Why

Target-set alignment diagnostic on B_smoke_epoch_12 (current best, mean 8,043 policy-only):

| metric | value | implication |
|---|---|---|
| V12 target top1_prob | 0.261 | what B was supposed to learn |
| **B prob @ target_top1** | **0.021** | what B actually outputs (12× too small) |
| B mass on target set | 0.097 | only 9.7% of mass on the 5 trained actions |
| B argmax = target argmax | 62.5% | direction right but not commitment |
| B's argmax rank (target_top1 vs B legal) | mean 1.6, P50 1.0 | usually B's top-1 or top-2 |

B has learned the ranking direction but distributes 90.3% of its probability mass on legal actions the V12 targets gave ~0% to. This is severe under-commitment to its own training targets.

Hypothesis: sharpening the V12 training targets via `--target-temperature` produces a peakier loss landscape that forces the model to commit. Decision criterion: ≥+5% over B_ep12 mean (≥8,450) with stable floor (P10 not worse).

## Three temperatures (ChatGPT-approved ladder)

| run | --target-temperature | effect on target | regime |
|---|---|---|---|
| **sharp_75** | 0.75 | top1 0.26 → 0.28 | conservative |
| **sharp_50** | 0.5  | top1 0.26 → 0.31 | moderate |
| **sharp_25** | 0.25 | top1 0.26 → 0.38 | aggressive |

T=0.1 is intentionally skipped: at T=0.1, 12% of targets become near-one-hot, replacing soft over-distillation with hard noise-distillation. If T=0.25 doesn't deliver, the answer is structural change, not stronger sharpening.

## Recipe (identical to B otherwise)

- Warm-start from B_smoke_epoch_12 (the current best)
- V12 corpus `selfplay.pt` (same as pillar2z / A / B / C / D)
- `--color-augment` (color permutation symmetry)
- 12 epochs, lr=3e-4, batch=32768, warmup=1
- `--oracle-lambda 0` — NO oracle. Single-variable test.

## Upload to Drive (`MyDrive/alphatrain/`)

1. `colorlines_path_b.tar.gz` — code (~1.1 MB; already there)
2. `v12_pillar2z.pt.gz` — V12 tensor (already there)
3. `path_b_B_smoke_best.pt` — B_ep12 warm-start (use Drive copy you saved during Path B v1 ablation)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_path_b.tar.gz /content/
!cd /content && tar xzf colorlines_path_b.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/path_b_B_smoke_best.pt',
            '/content/alphatrain/data/b_smoke_epoch_12.pt')
print(f'B warm-start: {os.path.getsize("/content/alphatrain/data/b_smoke_epoch_12.pt")/1e6:.0f} MB')

t0 = time.time()
!gzip -dc {DRIVE}/v12_pillar2z.pt.gz > /content/alphatrain/data/selfplay.pt
print(f'V12 tensor: {os.path.getsize("/content/alphatrain/data/selfplay.pt")/1e9:.1f} GB ({time.time()-t0:.0f}s)')

!pip install -q numpy numba scipy pytest

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

%cd /content
# Verify target-temperature diagnostic on B BEFORE training (sanity check)
!python -m alphatrain.scripts.analyze_target_alignment \
    --checkpoint alphatrain/data/b_smoke_epoch_12.pt \
    --tensor-file alphatrain/data/selfplay.pt \
    --n-samples 2000 --device cuda

## Run sharp_75 (T=0.75 — conservative)

~3-4h on H100/L4 (lighter than full Path B runs since no oracle batch concat).

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/selfplay.pt \
    --amp --compile \
    --resume alphatrain/data/b_smoke_epoch_12.pt --warm-start \
    --color-augment \
    --epochs 12 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --oracle-lambda 0.0 \
    --target-temperature 0.75 \
    --copy-to /content/drive/MyDrive/alphatrain/sharp_75_best.pt \
    --save-dir /content/checkpoints/sharp_75

In [ ]:
# Diagnose sharp_75 at every epoch
import glob
for ckpt in sorted(glob.glob('/content/checkpoints/sharp_75/epoch_*.pt')):
    print(f'\n===== {ckpt} =====')
    !python -m alphatrain.scripts.analyze_target_alignment \
        --checkpoint {ckpt} \
        --tensor-file alphatrain/data/selfplay.pt \
        --n-samples 2000 --device cuda 2>&1 | grep -E 'epoch|prob @ target|mass on target|argmax|rank.*legal'

# Copy epoch checkpoints to Drive
import shutil, os
DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/sharp_75/epoch_*.pt')):
    dst = f'{DRIVE}/sharp_75_{os.path.basename(f)}'
    shutil.copy(f, dst)

## Run sharp_50 (T=0.5 — moderate)

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/selfplay.pt \
    --amp --compile \
    --resume alphatrain/data/b_smoke_epoch_12.pt --warm-start \
    --color-augment \
    --epochs 12 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --oracle-lambda 0.0 \
    --target-temperature 0.5 \
    --copy-to /content/drive/MyDrive/alphatrain/sharp_50_best.pt \
    --save-dir /content/checkpoints/sharp_50

In [ ]:
import glob, shutil, os
for ckpt in sorted(glob.glob('/content/checkpoints/sharp_50/epoch_*.pt')):
    print(f'\n===== {ckpt} =====')
    !python -m alphatrain.scripts.analyze_target_alignment \
        --checkpoint {ckpt} \
        --tensor-file alphatrain/data/selfplay.pt \
        --n-samples 2000 --device cuda 2>&1 | grep -E 'epoch|prob @ target|mass on target|argmax|rank.*legal'

DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/sharp_50/epoch_*.pt')):
    dst = f'{DRIVE}/sharp_50_{os.path.basename(f)}'
    shutil.copy(f, dst)

## Run sharp_25 (T=0.25 — aggressive)

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/selfplay.pt \
    --amp --compile \
    --resume alphatrain/data/b_smoke_epoch_12.pt --warm-start \
    --color-augment \
    --epochs 12 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --oracle-lambda 0.0 \
    --target-temperature 0.25 \
    --copy-to /content/drive/MyDrive/alphatrain/sharp_25_best.pt \
    --save-dir /content/checkpoints/sharp_25

In [ ]:
import glob, shutil, os
for ckpt in sorted(glob.glob('/content/checkpoints/sharp_25/epoch_*.pt')):
    print(f'\n===== {ckpt} =====')
    !python -m alphatrain.scripts.analyze_target_alignment \
        --checkpoint {ckpt} \
        --tensor-file alphatrain/data/selfplay.pt \
        --n-samples 2000 --device cuda 2>&1 | grep -E 'epoch|prob @ target|mass on target|argmax|rank.*legal'

DRIVE = '/content/drive/MyDrive/alphatrain'
for f in sorted(glob.glob('/content/checkpoints/sharp_25/epoch_*.pt')):
    dst = f'{DRIVE}/sharp_25_{os.path.basename(f)}'
    shutil.copy(f, dst)

## Post-run analysis (download to M5 and eval there)

After this notebook completes, download `sharp_{75,50,25}_epoch_{1..12}.pt` from Drive to local. Then run on M5:

```bash
# Aggregate distribution + gameplay metrics across all 36 checkpoints
python -m alphatrain.scripts.overnight_analysis \
    --checkpoints \
        sharp_75_epoch_1 sharp_75_epoch_3 sharp_75_epoch_5 sharp_75_epoch_7 sharp_75_epoch_9 sharp_75_epoch_12 \
        sharp_50_epoch_1 sharp_50_epoch_3 sharp_50_epoch_5 sharp_50_epoch_7 sharp_50_epoch_9 sharp_50_epoch_12 \
        sharp_25_epoch_1 sharp_25_epoch_3 sharp_25_epoch_5 sharp_25_epoch_7 sharp_25_epoch_9 sharp_25_epoch_12 \
    --output alphatrain/data/sharpening_results.json \
    --seeds 500
```

Decision matrix:

| | sharp ≥+5% over B | sharp ≈ B | sharp regresses |
|---|---|---|---|
| **T=0.5 / 0.25 best** | Sharpening is the fix — adopt as new pipeline default | Flatness wasn't bottleneck; investigate structural | Sharpening was wrong; do NOT add hard CE on top |
| **T=0.75 best, 0.5/0.25 hurt floor** | Conservative sharpening; sweet spot found | — | — |
